# WiDS Wildfire Survival: Structural Gate + Local Timing Ensemble


In [1]:
import os, warnings, random, math
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
warnings.filterwarnings("ignore")

RUN_MODE = "full"   # use "fast" only for quick local smoke tests
OUTPUT_PATH = "/kaggle/working/submission.csv" if os.path.isdir("/kaggle/working") else "submission.csv"
CV_SEEDS = (17, 43, 91, 137, 211) if RUN_MODE == "full" else (17, 43, 91)
MODEL_TREES = 110 if RUN_MODE == "full" else 55
RANDOM_SEARCH_N = 14000 if RUN_MODE == "full" else 2500

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge, BayesianRidge, HuberRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import pairwise_distances


def find_data_dir():
    need = {"train.csv", "test.csv", "sample_submission.csv"}
    direct = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide_globaldathon26",
        "/kaggle/input",
        ".",
        "/mnt/data",
    ]
    for d in direct:
        if os.path.isdir(d) and need.issubset(set(os.listdir(d))):
            return d
    for root in ["/kaggle/input", "/mnt/data", "."]:
        if not os.path.isdir(root):
            continue
        for path, _, files in os.walk(root):
            if need.issubset(set(files)):
                return path
    raise FileNotFoundError("Could not find train.csv, test.csv, sample_submission.csv")


DATA_DIR = find_data_dir()
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

ID = "event_id"
TARGET_TIME = "time_to_hit_hours"
TARGET_EVENT = "event"
HORIZONS = [12, 24, 48]
P_COLS = ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]

time_values = train[TARGET_TIME].to_numpy(float)
event_values = train[TARGET_EVENT].to_numpy(int)
near_train = train["dist_min_ci_0_5h"].to_numpy(float) < 5000.0
near_test = test["dist_min_ci_0_5h"].to_numpy(float) < 5000.0
positive_idx = np.where(event_values == 1)[0]
pos_df = train.iloc[positive_idx].copy().reset_index(drop=True)
pos_time = train.iloc[positive_idx][TARGET_TIME].to_numpy(float)
Y_POS = np.column_stack([(pos_time <= h).astype(int) for h in HORIZONS])
GLOBAL_RATES = Y_POS.mean(axis=0)


def enforce_monotone(arr):
    arr = np.clip(np.asarray(arr, dtype=float), 0.0, 1.0)
    for j in range(1, arr.shape[1]):
        arr[:, j] = np.maximum(arr[:, j], arr[:, j - 1])
    return arr


def c_index(time, event, risk):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    risk = np.asarray(risk, dtype=float)
    comparable = (event[:, None] == 1) & (time[:, None] < time[None, :])
    den = comparable.sum()
    if den == 0:
        return 0.5
    diff = risk[:, None] - risk[None, :]
    conc = ((diff > 0).astype(float) + 0.5 * (diff == 0).astype(float))[comparable].sum()
    return float(conc / den)


def brier_at(time, event, prob, horizon):
    valid = ~((event == 0) & (time < horizon))
    y = ((event == 1) & (time <= horizon)).astype(float)[valid]
    p = np.clip(prob[valid], 0.0, 1.0)
    return float(np.mean((p - y) ** 2))


def hybrid_score(pred):
    pred = enforce_monotone(pred)
    risk = 0.3 * pred[:, 1] + 0.4 * pred[:, 2] + 0.3 * pred[:, 3]
    ci = c_index(time_values, event_values, risk)
    b24 = brier_at(time_values, event_values, pred[:, 1], 24)
    b48 = brier_at(time_values, event_values, pred[:, 2], 48)
    b72 = brier_at(time_values, event_values, pred[:, 3], 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * ci + 0.7 * (1.0 - wb), ci, wb, b24, b48, b72


def wrap_positive_candidate(pos_oof, test_conditional, name):
    pos_oof = np.asarray(pos_oof, dtype=float)
    test_conditional = np.asarray(test_conditional, dtype=float)
    train_pred = np.zeros((len(train), 4), dtype=float)
    test_pred = np.zeros((len(test), 4), dtype=float)
    train_pred[:, 3] = 1.0
    test_pred[:, 3] = 1.0
    train_pred[positive_idx, :3] = pos_oof
    test_pred[:, :3] = test_conditional
    train_pred[~near_train, :3] = 0.0
    test_pred[~near_test, :3] = 0.0
    return name, enforce_monotone(train_pred), enforce_monotone(test_pred)


def make_features(df):
    r = df.copy()
    dist = np.maximum(r["dist_min_ci_0_5h"].to_numpy(float), 1.0)
    area = np.maximum(r["area_first_ha"].to_numpy(float), 0.0)
    perim = r["num_perimeters_0_5h"].to_numpy(float)
    low = r["low_temporal_resolution_0_5h"].to_numpy(float)
    align = r["alignment_abs"].to_numpy(float)
    hr = r["event_start_hour"].to_numpy(float)
    mo = r["event_start_month"].to_numpy(float)
    dow = r["event_start_dayofweek"].to_numpy(float)
    growth = np.maximum(r["radial_growth_rate_m_per_h"].to_numpy(float), 0.0) + np.maximum(r["closing_speed_m_per_h"].to_numpy(float), 0.0)
    radius = np.sqrt(area * 10000.0 / np.pi)

    r["dist_km"] = dist / 1000.0
    r["log_dist"] = np.log1p(dist)
    r["inside5k_km"] = (5000.0 - dist) / 1000.0
    r["inv_dist"] = 1.0 / (dist / 1000.0 + 0.10)
    r["dist_to_5k"] = dist - 5000.0
    r["log_area"] = np.log1p(area)
    r["sqrt_area"] = np.sqrt(area)
    r["fire_radius_m"] = radius
    r["radius_to_dist"] = radius / dist
    r["perim_log"] = np.log1p(perim)
    r["eff_growth"] = growth
    r["log_eff_growth"] = np.log1p(growth)
    r["low_single"] = ((low == 1.0) & (perim <= 1.0)).astype(float)
    r["multi_good"] = ((perim >= 2.0) & (low == 0.0)).astype(float)
    r["align_hi"] = (align > 0.70).astype(float)
    r["align_mid"] = (align > 0.40).astype(float)
    r["low_tiny"] = r["low_single"] * (area < 10.0).astype(float)
    r["low_small"] = r["low_single"] * (area < 30.0).astype(float)
    r["low_large"] = r["low_single"] * (area > 500.0).astype(float)

    for th in [1, 2, 3, 5, 10, 15]:
        r[f"perim_ge_{th}"] = (perim >= th).astype(float)
    for th in [5, 10, 20, 30, 50, 100, 250, 500, 1000, 1500]:
        r[f"area_lt_{th}"] = (area < th).astype(float)
    for th in [500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500]:
        r[f"dist_lt_{th}"] = (dist < th).astype(float)

    for period, values, prefix in [(24.0, hr, "hour"), (12.0, mo, "month"), (7.0, dow, "dow")]:
        r[f"{prefix}_sin"] = np.sin(2.0 * np.pi * values / period)
        r[f"{prefix}_cos"] = np.cos(2.0 * np.pi * values / period)
    for a, b, nm in [(0, 5, "night"), (6, 11, "morning"), (12, 15, "afternoon"), (16, 19, "late_afternoon"), (18, 22, "evening"), (20, 23, "late")]:
        r[nm] = ((hr >= a) & (hr <= b)).astype(float)
    for m in [5, 6, 7, 8, 9]:
        r[f"month_{m}"] = (mo == m).astype(float)

    r["align_growth"] = align * r["log_eff_growth"]
    r["align_inv_dist"] = align * r["inv_dist"]
    r["area_dist"] = r["log_area"] * r["inv_dist"]
    r["perim_area"] = r["perim_log"] * r["log_area"]
    r["perim_dist"] = r["perim_log"] * r["inv_dist"]
    r["late_signature"] = r["low_single"] * ((area < 30.0).astype(float) + 0.40 * ((hr >= 14.0) & (hr <= 18.0)).astype(float) + 0.20 * (mo == 7.0).astype(float))
    r["fast_signature"] = r["multi_good"] + 0.70 * (perim >= 5.0).astype(float) + 0.40 * r["align_hi"] + 0.25 * (growth > 1.0).astype(float) + 0.10 * np.log1p(area) - 0.70 * r["low_single"]

    return r.replace([np.inf, -np.inf], np.nan).fillna(0.0)


train_fe = make_features(train)
test_fe = make_features(test)
feature_cols = [c for c in train_fe.columns if c not in [ID, TARGET_TIME, TARGET_EVENT]]
X_all = train_fe[feature_cols]
X_test = test_fe[feature_cols]
X_pos = X_all.iloc[positive_idx].reset_index(drop=True)


candidate_names, candidate_train, candidate_test = [], [], []


def add_candidate(cand):
    name, trp, tep = cand
    candidate_names.append(name)
    candidate_train.append(trp)
    candidate_test.append(tep)


# Baseline: leave-one-out empirical CDF among known hits.
add_candidate(wrap_positive_candidate((Y_POS.sum(axis=0) - Y_POS) / max(1, len(Y_POS) - 1), np.tile(GLOBAL_RATES, (len(test), 1)), "global_loo_cdf"))


# Deterministic physics/timing score: designed for the structural <5km hit gate.
def physics_conditional(df):
    dist = df["dist_min_ci_0_5h"].to_numpy(float)
    area = np.maximum(df["area_first_ha"].to_numpy(float), 0.0)
    perim = df["num_perimeters_0_5h"].to_numpy(float)
    low = df["low_temporal_resolution_0_5h"].to_numpy(float)
    align = df["alignment_abs"].to_numpy(float)
    hr = df["event_start_hour"].to_numpy(float)
    growth = np.maximum(df["radial_growth_rate_m_per_h"].to_numpy(float), 0.0) + np.maximum(df["closing_speed_m_per_h"].to_numpy(float), 0.0)
    s = np.zeros(len(df), dtype=float)
    s += 1.08 * (perim >= 2.0)
    s += 0.92 * (perim >= 3.0)
    s += 0.90 * (perim >= 5.0)
    s += 0.30 * np.log1p(area)
    s += 0.55 * (align > 0.50)
    s += 0.35 * (align > 0.85)
    s += 0.45 * (growth > 1.0)
    s += 0.26 * (dist < 1200.0)
    s += 0.10 * (dist < 700.0)
    s -= 0.11 * (dist > 3200.0)
    s -= 0.18 * (dist > 4500.0)
    s -= 1.05 * low
    s -= 0.58 * (area < 30.0)
    s -= 0.42 * (area < 10.0)
    s -= 0.12 * ((hr <= 5.0) | (hr >= 23.0))
    s += 0.18 * ((hr >= 18.0) & (hr <= 22.0))
    s -= 0.20 * ((low == 1.0) & (perim <= 1.0) & (area < 20.0) & (hr >= 14.0) & (hr <= 18.0))
    return np.column_stack([expit(s - 0.28), expit(s + 1.02), expit(s + 1.88)])


add_candidate(wrap_positive_candidate(physics_conditional(pos_df), physics_conditional(test), "physics_structural_timing"))


# Hierarchical smoothed group CDFs.
def group_key(df, spec):
    p = df["num_perimeters_0_5h"].to_numpy(float)
    low = df["low_temporal_resolution_0_5h"].to_numpy(float)
    area = df["area_first_ha"].to_numpy(float)
    dist = df["dist_min_ci_0_5h"].to_numpy(float)
    hr = df["event_start_hour"].to_numpy(float)
    mo = df["event_start_month"].to_numpy(float)
    align = df["alignment_abs"].to_numpy(float)
    parts = []
    for token in spec:
        if token == "lp":
            parts.append(np.where((low == 1.0) & (p <= 1.0), "L1", np.where(p >= 5.0, "P5", np.where(p >= 2.0, "P2", "O"))))
        elif token == "ab":
            parts.append(pd.cut(area, [-1, 10, 30, 100, 500, 1e9], labels=False).astype(str))
        elif token == "db":
            parts.append(pd.cut(dist, [-1, 1000, 2000, 3000, 4000, 5000], labels=False).astype(str))
        elif token == "hb":
            parts.append(pd.cut(hr, [-1, 5, 11, 15, 19, 23], labels=False).astype(str))
        elif token == "mb":
            parts.append(pd.cut(mo, [0, 5, 6, 7, 8, 12], labels=False).astype(str))
        elif token == "al":
            parts.append(pd.cut(align, [-0.01, 0.10, 0.50, 0.80, 1.01], labels=False).astype(str))
    return np.array(["|".join(x) for x in zip(*parts)]) if parts else np.array(["all"] * len(df))


def group_candidate(spec, alpha):
    kp = group_key(pos_df, spec)
    kt = group_key(test, spec)
    full_rates = {}
    for g in np.unique(kp):
        m = kp == g
        full_rates[g] = (Y_POS[m].sum(axis=0) + alpha * GLOBAL_RATES) / (m.sum() + alpha)
    pos_oof = np.zeros_like(Y_POS, dtype=float)
    for i, g in enumerate(kp):
        m = kp == g
        m[i] = False
        if m.sum() > 0:
            pos_oof[i] = (Y_POS[m].sum(axis=0) + alpha * GLOBAL_RATES) / (m.sum() + alpha)
        else:
            pos_oof[i] = GLOBAL_RATES
    test_cond = np.vstack([full_rates.get(g, GLOBAL_RATES) for g in kt])
    return wrap_positive_candidate(pos_oof, test_cond, "group_" + "_".join(spec) + f"_a{alpha:g}")


for spec in [("lp",), ("lp", "ab"), ("lp", "hb"), ("lp", "mb"), ("lp", "ab", "hb"), ("lp", "ab", "mb"), ("lp", "db"), ("lp", "al")]:
    for alpha in [1.0, 3.0, 8.0, 15.0]:
        add_candidate(group_candidate(spec, alpha))


# Local kernel CDFs: the main non-linear small-data model.
def local_basis(df, variant):
    dist = np.maximum(df["dist_min_ci_0_5h"].to_numpy(float), 1.0)
    area = np.maximum(df["area_first_ha"].to_numpy(float), 0.0)
    perim = df["num_perimeters_0_5h"].to_numpy(float)
    low = df["low_temporal_resolution_0_5h"].to_numpy(float)
    align = df["alignment_abs"].to_numpy(float)
    hr = df["event_start_hour"].to_numpy(float)
    mo = df["event_start_month"].to_numpy(float)
    dow = df["event_start_dayofweek"].to_numpy(float)
    growth = np.maximum(df["radial_growth_rate_m_per_h"].to_numpy(float), 0.0) + np.maximum(df["closing_speed_m_per_h"].to_numpy(float), 0.0)
    cols = [
        np.log1p(dist),
        (5000.0 - dist) / 1000.0,
        np.log1p(area),
        np.sqrt(area),
        np.log1p(perim),
        low,
        ((low == 1.0) & (perim <= 1.0)).astype(float),
        ((perim >= 2.0) & (low == 0.0)).astype(float),
        align,
        np.log1p(growth),
        np.sin(2.0 * np.pi * hr / 24.0),
        np.cos(2.0 * np.pi * hr / 24.0),
        np.sin(2.0 * np.pi * mo / 12.0),
        np.cos(2.0 * np.pi * mo / 12.0),
        np.sin(2.0 * np.pi * dow / 7.0),
        np.cos(2.0 * np.pi * dow / 7.0),
    ]
    if variant >= 1:
        cols += [
            (area < 10.0).astype(float), (area < 30.0).astype(float), (area > 500.0).astype(float),
            ((hr >= 16.0) & (hr <= 19.0)).astype(float), ((hr >= 20.0) & (hr <= 23.0)).astype(float),
            (mo == 7.0).astype(float), (mo == 8.0).astype(float), (dist < 1000.0).astype(float), (dist > 3500.0).astype(float),
        ]
    if variant >= 2:
        cols += [
            (perim >= 5.0).astype(float), (perim >= 10.0).astype(float),
            align * np.log1p(growth), np.log1p(area) / (dist / 1000.0 + 0.20),
            ((low == 1.0) & (perim <= 1.0) & (area < 30.0)).astype(float),
            ((low == 1.0) & (perim <= 1.0) & (hr >= 14.0) & (hr <= 18.0)).astype(float),
        ]
    return np.vstack([np.asarray(c, dtype=float) for c in cols]).T


def kernel_candidate(variant, bandwidth, shrink, power):
    B = local_basis(pos_df, variant)
    Bt = local_basis(test, variant)
    mu = B.mean(axis=0)
    sd = B.std(axis=0) + 1e-6
    Z = (B - mu) / sd
    Zt = (Bt - mu) / sd
    D = pairwise_distances(Z, Z, metric="euclidean")
    if power == 1:
        W = np.exp(-(D / bandwidth))
    else:
        W = np.exp(-0.5 * (D / bandwidth) ** 2)
    np.fill_diagonal(W, 0.0)
    den = W.sum(axis=1)
    eff = den ** 2 / (np.square(W).sum(axis=1) + 1e-12)
    lam = eff / (eff + shrink)
    raw = (W @ Y_POS) / np.maximum(den[:, None], 1e-12)
    pos_oof = lam[:, None] * raw + (1.0 - lam[:, None]) * GLOBAL_RATES

    Dt = pairwise_distances(Zt, Z, metric="euclidean")
    if power == 1:
        Wt = np.exp(-(Dt / bandwidth))
    else:
        Wt = np.exp(-0.5 * (Dt / bandwidth) ** 2)
    den_t = Wt.sum(axis=1)
    eff_t = den_t ** 2 / (np.square(Wt).sum(axis=1) + 1e-12)
    lam_t = eff_t / (eff_t + shrink)
    raw_t = (Wt @ Y_POS) / np.maximum(den_t[:, None], 1e-12)
    test_cond = lam_t[:, None] * raw_t + (1.0 - lam_t[:, None]) * GLOBAL_RATES
    return wrap_positive_candidate(np.clip(pos_oof, 0.0, 1.0), np.clip(test_cond, 0.0, 1.0), f"kernel_v{variant}_bw{bandwidth:g}_sh{shrink:g}_p{power}")


for variant in [0, 1, 2]:
    for bandwidth in [0.75, 1.20, 1.80, 2.60]:
        for shrink in [1.0, 3.0, 8.0]:
            for power in [1, 2]:
                add_candidate(kernel_candidate(variant, bandwidth, shrink, power))


# Repeated-CV model candidates on the positive set only.
bins = np.digitize(pos_time, [12.0, 24.0, 48.0])
min_bin = int(np.bincount(bins)[np.bincount(bins) > 0].min())
N_SPLITS = max(2, min(4, min_bin))


def iter_splits():
    for seed in CV_SEEDS:
        cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        yield from cv.split(X_pos, bins)


def positive_classifier_candidate(builder, name):
    oof = np.zeros((len(pos_df), 3), dtype=float)
    cnt = np.zeros(len(pos_df), dtype=float)
    test_acc = np.zeros((len(test), 3), dtype=float)
    for tr_idx, va_idx in iter_splits():
        cnt[va_idx] += 1.0
        for j, h in enumerate(HORIZONS):
            y = Y_POS[:, j]
            if len(np.unique(y[tr_idx])) < 2:
                oof[va_idx, j] += y[tr_idx].mean()
            else:
                model = builder(j, h, 17)
                model.fit(X_pos.iloc[tr_idx], y[tr_idx])
                oof[va_idx, j] += model.predict_proba(X_pos.iloc[va_idx])[:, 1]
    oof /= np.maximum(cnt[:, None], 1.0)
    for seed in CV_SEEDS:
        for j, h in enumerate(HORIZONS):
            y = Y_POS[:, j]
            model = builder(j, h, seed)
            model.fit(X_pos, y)
            test_acc[:, j] += model.predict_proba(X_test)[:, 1]
    test_acc /= len(CV_SEEDS)
    return wrap_positive_candidate(oof, test_acc, name)


def positive_regression_candidate(builder, name):
    y_log = np.log1p(pos_time)
    oof = np.zeros((len(pos_df), 3), dtype=float)
    cnt = np.zeros(len(pos_df), dtype=float)
    test_acc = np.zeros((len(test), 3), dtype=float)
    for tr_idx, va_idx in iter_splits():
        model = builder(17)
        model.fit(X_pos.iloc[tr_idx], y_log[tr_idx])
        mu_va = model.predict(X_pos.iloc[va_idx])
        mu_tr = model.predict(X_pos.iloc[tr_idx])
        resid = y_log[tr_idx] - mu_tr
        for j, h in enumerate(HORIZONS):
            z = np.log1p(h) - mu_va
            oof[va_idx, j] += np.mean(resid[None, :] <= z[:, None], axis=1)
        cnt[va_idx] += 1.0
    oof /= np.maximum(cnt[:, None], 1.0)
    for seed in CV_SEEDS:
        model = builder(seed)
        model.fit(X_pos, y_log)
        mu_test = model.predict(X_test)
        mu_train = model.predict(X_pos)
        resid = y_log - mu_train
        for j, h in enumerate(HORIZONS):
            z = np.log1p(h) - mu_test
            test_acc[:, j] += np.mean(resid[None, :] <= z[:, None], axis=1)
    test_acc /= len(CV_SEEDS)
    return wrap_positive_candidate(oof, test_acc, name)


classifier_builders = [
    ("logit_c016", lambda j, h, seed: Pipeline([("scale", RobustScaler()), ("lr", LogisticRegression(C=0.16, solver="liblinear", random_state=seed, max_iter=3000))])),
    ("logit_bal_c035", lambda j, h, seed: Pipeline([("scale", RobustScaler()), ("lr", LogisticRegression(C=0.35, solver="liblinear", class_weight="balanced", random_state=seed, max_iter=3000))])),
    ("extratrees_d2", lambda j, h, seed: ExtraTreesClassifier(n_estimators=MODEL_TREES, max_depth=2, min_samples_leaf=3, class_weight="balanced_subsample", random_state=seed, n_jobs=1)),
    ("extratrees_d3", lambda j, h, seed: ExtraTreesClassifier(n_estimators=MODEL_TREES, max_depth=3, min_samples_leaf=4, class_weight="balanced_subsample", random_state=seed, n_jobs=1)),
    ("gb_stump", lambda j, h, seed: GradientBoostingClassifier(n_estimators=75 if RUN_MODE == "full" else 40, learning_rate=0.040, max_depth=1, min_samples_leaf=5, subsample=0.85, random_state=seed)),
    ("knn7", lambda j, h, seed: Pipeline([("scale", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=7, weights="distance"))])),
]
for name, builder in classifier_builders:
    add_candidate(positive_classifier_candidate(builder, name))


regression_builders = [
    ("ridge8_rescdf", lambda seed: Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=8.0))])),
    ("bayes_rescdf", lambda seed: Pipeline([("scale", StandardScaler()), ("br", BayesianRidge())])),
    ("huber_rescdf", lambda seed: Pipeline([("scale", StandardScaler()), ("huber", HuberRegressor(alpha=0.70, epsilon=1.25, max_iter=1200))])),
    ("extratrees_reg_d3", lambda seed: ExtraTreesRegressor(n_estimators=MODEL_TREES, max_depth=3, min_samples_leaf=4, random_state=seed, n_jobs=1)),
    ("gb_reg_stump", lambda seed: GradientBoostingRegressor(n_estimators=75 if RUN_MODE == "full" else 40, learning_rate=0.040, max_depth=1, min_samples_leaf=5, subsample=0.85, random_state=seed)),
    ("knn7_rescdf", lambda seed: Pipeline([("scale", StandardScaler()), ("knn", KNeighborsRegressor(n_neighbors=7, weights="distance"))])),
]
for name, builder in regression_builders:
    add_candidate(positive_regression_candidate(builder, name))


candidate_train = np.stack(candidate_train, axis=0)
candidate_test = np.stack(candidate_test, axis=0)
scores = np.array([hybrid_score(candidate_train[i])[0] for i in range(len(candidate_names))], dtype=float)

print("DATA_DIR", DATA_DIR)
print("n_train", len(train), "n_test", len(test), "near_train", int(near_train.sum()), "near_test", int(near_test.sum()), "candidates", len(candidate_names))
print("Top single candidates:")
for i in np.argsort(-scores)[:18]:
    sc = hybrid_score(candidate_train[i])
    print(f"{i:03d} {candidate_names[i]:28s} score={sc[0]:.6f} c={sc[1]:.6f} wb={sc[2]:.6f}")


def blend_arrays(stack, weights):
    return enforce_monotone(np.tensordot(weights, stack, axes=(0, 0)))


# Non-smooth metric blending with a score-softmax prior.
temperature = 0.0035
prior = np.exp((scores - scores.max()) / temperature)
prior /= prior.sum()
order = np.argsort(-scores)
best_w = prior.copy()
best_score = hybrid_score(blend_arrays(candidate_train, best_w))[0]
for k in [1, 2, 3, 5, 8, 13, 21, min(34, len(candidate_names))]:
    w = np.zeros(len(candidate_names), dtype=float)
    w[order[:min(k, len(candidate_names))]] = 1.0 / min(k, len(candidate_names))
    sc = hybrid_score(blend_arrays(candidate_train, w))[0]
    if sc > best_score:
        best_score, best_w = sc, w.copy()

rng = np.random.default_rng(20260425)
alpha_center = 1.0 + 50.0 * prior
for _ in range(RANDOM_SEARCH_N):
    if rng.random() < 0.65:
        w = rng.dirichlet(alpha_center)
    else:
        k = int(rng.integers(2, min(12, len(candidate_names)) + 1))
        chosen = rng.choice(order[:min(30, len(candidate_names))], size=k, replace=False)
        w = np.zeros(len(candidate_names), dtype=float)
        w[chosen] = rng.dirichlet(np.ones(k) * 1.20)
    sc = hybrid_score(blend_arrays(candidate_train, w))[0]
    if sc > best_score:
        best_score, best_w = sc, w.copy()

# Shrink the hill-climbed blend toward the performance prior; selected by OOF, not fixed by LB.
gamma_grid = [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60]
best_final_w = best_w.copy()
best_final_score = hybrid_score(blend_arrays(candidate_train, best_final_w))[0]
for gamma in gamma_grid:
    w = (1.0 - gamma) * best_w + gamma * prior
    w /= w.sum()
    sc = hybrid_score(blend_arrays(candidate_train, w))[0]
    if sc > best_final_score:
        best_final_score = sc
        best_final_w = w.copy()

blend_train = blend_arrays(candidate_train, best_final_w)
blend_test = blend_arrays(candidate_test, best_final_w)

# Horizon-wise isotonic calibration, constrained by greedy OOF improvement.
for j, h in enumerate(HORIZONS):
    valid = ~((event_values == 0) & (time_values < h))
    y = ((event_values == 1) & (time_values <= h)).astype(float)
    if len(np.unique(y[valid])) < 2:
        continue
    ir = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
    ir.fit(blend_train[valid, j], y[valid])
    orig_train = blend_train[:, j].copy()
    orig_test = blend_test[:, j].copy()
    local_best = hybrid_score(blend_train)[0]
    local_train = blend_train.copy()
    local_test = blend_test.copy()
    for a in np.linspace(0.0, 0.45, 10):
        tr_try = blend_train.copy()
        te_try = blend_test.copy()
        tr_try[:, j] = (1.0 - a) * orig_train + a * ir.predict(orig_train)
        te_try[:, j] = (1.0 - a) * orig_test + a * ir.predict(orig_test)
        tr_try = enforce_monotone(tr_try)
        te_try = enforce_monotone(te_try)
        tr_try[~near_train, :3] = 0.0
        te_try[~near_test, :3] = 0.0
        tr_try[:, 3] = 1.0
        te_try[:, 3] = 1.0
        sc = hybrid_score(tr_try)[0]
        if sc > local_best + 1e-12:
            local_best = sc
            local_train = tr_try
            local_test = te_try
    blend_train, blend_test = local_train, local_test


# Optional schedule/low-resolution correction: only allowed to lower single-perimeter low-res rows when OOF improves.
def schedule_basis_frame(df):
    hr = df["event_start_hour"].to_numpy(float)
    mo = df["event_start_month"].to_numpy(float)
    dow = df["event_start_dayofweek"].to_numpy(float)
    area = np.maximum(df["area_first_ha"].to_numpy(float), 0.0)
    dist = np.maximum(df["dist_min_ci_0_5h"].to_numpy(float), 1.0)
    return np.nan_to_num(np.vstack([
        np.sin(2.0 * np.pi * hr / 24.0), np.cos(2.0 * np.pi * hr / 24.0),
        np.sin(2.0 * np.pi * mo / 12.0), np.cos(2.0 * np.pi * mo / 12.0),
        np.sin(2.0 * np.pi * dow / 7.0), np.cos(2.0 * np.pi * dow / 7.0),
        np.log1p(area), np.log1p(dist), (area < 10.0).astype(float), (area < 30.0).astype(float),
        ((hr >= 16.0) & (hr <= 19.0)).astype(float), ((hr >= 20.0) & (hr <= 23.0)).astype(float),
    ]).T.astype(float), copy=False)


def lowres_schedule_oof_and_test(bandwidth=1.0, shrink=1.2):
    sched_pos = ((pos_df["low_temporal_resolution_0_5h"].to_numpy(float) == 1.0) &
                 (pos_df["num_perimeters_0_5h"].to_numpy(float) <= 1.0))
    sched_test = ((test["low_temporal_resolution_0_5h"].to_numpy(float) == 1.0) &
                  (test["num_perimeters_0_5h"].to_numpy(float) <= 1.0))
    pos_oof = np.zeros_like(Y_POS, dtype=float)
    nons_pos = ~sched_pos
    if nons_pos.sum() >= 2:
        s = Y_POS[nons_pos].sum(axis=0)
        n = int(nons_pos.sum())
        for i in np.where(nons_pos)[0]:
            pos_oof[i] = (s - Y_POS[i]) / max(1, n - 1)
    else:
        pos_oof[nons_pos] = GLOBAL_RATES
    if sched_pos.sum() >= 3:
        B = schedule_basis_frame(pos_df.loc[sched_pos].reset_index(drop=True))
        Bt = schedule_basis_frame(test)
        mu = B.mean(axis=0)
        sd = B.std(axis=0) + 1e-6
        Z = (B - mu) / sd
        Zt = (Bt - mu) / sd
        Ys = Y_POS[sched_pos]
        idx = np.where(sched_pos)[0]
        D = np.sum((Z[:, None, :] - Z[None, :, :]) ** 2, axis=2)
        for a, i in enumerate(idx):
            W = np.exp(-0.5 * D[a] / (bandwidth ** 2))
            W[a] = 0.0
            den = W.sum()
            eff = den ** 2 / (np.square(W).sum() + 1e-12)
            lam = eff / (eff + shrink)
            raw = (W @ Ys) / max(den, 1e-12)
            pos_oof[i] = lam * raw + (1.0 - lam) * GLOBAL_RATES
        Dt = np.sum((Zt[:, None, :] - Z[None, :, :]) ** 2, axis=2)
        Wt = np.exp(-0.5 * Dt / (bandwidth ** 2))
        den = Wt.sum(axis=1)
        eff = den ** 2 / (np.square(Wt).sum(axis=1) + 1e-12)
        lam = eff / (eff + shrink)
        raw = (Wt @ Ys) / np.maximum(den[:, None], 1e-12)
        test_rates = lam[:, None] * raw + (1.0 - lam[:, None]) * GLOBAL_RATES[None, :]
    else:
        test_rates = np.tile(GLOBAL_RATES, (len(test), 1))
    return np.clip(pos_oof, 0.0, 1.0), np.clip(test_rates, 0.0, 1.0), sched_test


def apply_schedule_min_blend(base_train, base_test, alpha=0.12, bandwidth=1.0, shrink=1.2):
    pos_oof, test_rates, sched_test = lowres_schedule_oof_and_test(bandwidth=bandwidth, shrink=shrink)
    sched_train = np.zeros_like(base_train)
    sched_test_pred = np.zeros_like(base_test)
    sched_train[:, 3] = 1.0
    sched_test_pred[:, 3] = 1.0
    sched_train[positive_idx, :3] = pos_oof
    sched_test_pred[:, :3] = test_rates
    train_mask = near_train & (train["low_temporal_resolution_0_5h"].to_numpy(float) == 1.0) & (train["num_perimeters_0_5h"].to_numpy(float) <= 1.0)
    test_mask = near_test & sched_test
    out_train = base_train.copy()
    out_test = base_test.copy()
    out_train[train_mask, :3] = (1.0 - alpha) * out_train[train_mask, :3] + alpha * np.minimum(out_train[train_mask, :3], sched_train[train_mask, :3])
    out_test[test_mask, :3] = (1.0 - alpha) * out_test[test_mask, :3] + alpha * np.minimum(out_test[test_mask, :3], sched_test_pred[test_mask, :3])
    out_train[~near_train, :3] = 0.0
    out_test[~near_test, :3] = 0.0
    out_train[:, 3] = 1.0
    out_test[:, 3] = 1.0
    return enforce_monotone(out_train), enforce_monotone(out_test)


best_train = blend_train.copy()
best_test = blend_test.copy()
best_post_score = hybrid_score(best_train)[0]
for bw in [0.70, 0.90, 1.10, 1.40, 1.80]:
    for sh in [0.8, 1.2, 2.0, 3.5]:
        for alpha in [0.00, 0.04, 0.08, 0.12, 0.16, 0.22, 0.30]:
            tr_try, te_try = apply_schedule_min_blend(blend_train, blend_test, alpha=alpha, bandwidth=bw, shrink=sh)
            sc = hybrid_score(tr_try)[0]
            if sc > best_post_score + 1e-12:
                best_post_score = sc
                best_train, best_test = tr_try, te_try
blend_train, blend_test = enforce_monotone(best_train), enforce_monotone(best_test)

# Final structural constraints.
blend_train[~near_train, :3] = 0.0
blend_test[~near_test, :3] = 0.0
blend_train[:, 3] = 1.0
blend_test[:, 3] = 1.0
blend_train = enforce_monotone(blend_train)
blend_test = enforce_monotone(blend_test)

print("Final OOF", hybrid_score(blend_train))
print("Final nonzero weights:")
for nm, w, sc in sorted(zip(candidate_names, best_final_w, scores), key=lambda z: -z[1]):
    if w > 1e-4:
        print(f"  {w:8.5f}  {sc:.6f}  {nm}")

submission = sample[[ID]].copy()
pred_df = pd.DataFrame({ID: test[ID].values})
pred_df[P_COLS] = blend_test
submission = submission.merge(pred_df, on=ID, how="left")
submission[P_COLS] = submission[P_COLS].astype(float)
submission[P_COLS] = enforce_monotone(submission[P_COLS].to_numpy(float))
submission["prob_72h"] = 1.0
submission = submission[[ID] + P_COLS]

assert len(submission) == len(sample)
assert submission[ID].is_unique
assert set(submission[ID]) == set(sample[ID])
assert np.isfinite(submission[P_COLS].to_numpy(float)).all()
assert ((submission[P_COLS] >= 0.0) & (submission[P_COLS] <= 1.0)).all().all()
assert (submission[P_COLS].diff(axis=1).iloc[:, 1:] >= -1e-12).all().all()

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True) if os.path.dirname(OUTPUT_PATH) else None
submission.to_csv(OUTPUT_PATH, index=False)
print("saved", OUTPUT_PATH, submission.shape)
print(submission.describe())


DATA_DIR /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
n_train 221 n_test 95 near_train 69 near_test 28 candidates 118
Top single candidates:
001 physics_structural_timing    score=0.975408 c=0.946837 wb=0.012347
038 kernel_v0_bw0.75_sh8_p1      score=0.974952 c=0.948493 wb=0.013708
036 kernel_v0_bw0.75_sh3_p1      score=0.974758 c=0.946920 wb=0.013311
051 kernel_v0_bw1.8_sh8_p2       score=0.974242 c=0.947168 wb=0.014155
044 kernel_v0_bw1.2_sh8_p1       score=0.974179 c=0.946837 wb=0.014103
034 kernel_v0_bw0.75_sh1_p1      score=0.974128 c=0.944601 wb=0.013217
049 kernel_v0_bw1.8_sh3_p2       score=0.974125 c=0.946505 wb=0.014038
088 kernel_v2_bw1.2_sh1_p1       score=0.974118 c=0.937562 wb=0.010215
090 kernel_v2_bw1.2_sh3_p1       score=0.973980 c=0.939550 wb=0.011264
042 kernel_v0_bw1.2_sh3_p1       score=0.973979 c=0.945677 wb=0.013891
064 kernel_v1_bw1.2_sh1_p1       score=0.973950 c=0.939053 wb=0.011094
094 kernel_v2_bw1.8_sh1_p1       score=0.973877 c=0.939550 wb=0.011